# Starter Cells

Setup

In [1]:
from pathlib import Path
import polars as pl

ON_KAGGLE = Path("/kaggle/input").exists()
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = Path("/kaggle/input/nfl-big-data-bowl-2027") if ON_KAGGLE else ROOT / "data/nfl-big-data-bowl-2027"
CACHE = Path("/kaggle/working/parquet") if ON_KAGGLE else ROOT / "data/parquet"
CACHE.mkdir(parents=True, exist_ok=True)

READ = dict(null_values=["NA"], infer_schema_length=100_000)
TIME = pl.col("time").str.to_datetime(time_unit="ms")
FIELD_W = 160 / 3  # 53.33 yd


Small tables, combine tracking, player_play

In [2]:
for name in ["players", "combine_results", "player_career_successes", "games"]:
    pl.read_csv(RAW / f"{name}.csv", **READ).write_parquet(CACHE / f"{name}.parquet")

pl.scan_csv(RAW / "combine_tracking.csv", **READ).with_columns(TIME).sink_parquet(CACHE / "combine_tracking.parquet")

pp = (pl.read_csv(RAW / "player_play.csv", **READ)
        .rename({"visitor_team_win_probility_added": "visitor_team_win_probability_added"})
        .with_columns(pl.col(pl.String).replace("", None)))      # pass_result is "" (not NA) on runs
pp.write_parquet(CACHE / "player_play.parquet")

Play-level table with line of scrimmage

In [3]:
PLAY_COLS = ["game_id", "play_id", "possession_team", "play_direction", "yardline_side", "yardline_number",
             "quarter", "down", "yards_to_go", "game_clock", "offense_formation", "receiver_alignment",
             "pass_result", "pass_length", "team_coverage_man_zone", "team_coverage_type",
             "expected_points", "expected_points_added", "play_description"]
plays = (pp.group_by("game_id", "play_id")
           .agg(pl.col(c).drop_nulls().first() for c in PLAY_COLS[2:])
           .with_columns(los_x=pl.when(pl.col("yardline_number") == 50).then(60)
                                 .when(pl.col("yardline_side") == pl.col("possession_team")).then(10 + pl.col("yardline_number"))
                                 .otherwise(110 - pl.col("yardline_number"))))
plays.write_parquet(CACHE / "plays.parquet")


Aligned game tracking (22.6M rows → ~700 MB Parquet)

In [4]:
gt = pl.concat([pl.scan_csv(RAW / f"game_tracking_{yr}.csv", **READ).with_columns(TIME, season=pl.lit(yr))
                for yr in (2023, 2024, 2025)])
flip = pl.col("play_direction") == "left"
aligned = (gt.join(plays.lazy().select("game_id", "play_id", "play_direction", "los_x"), on=["game_id", "play_id"], how="left")
             .with_columns(snap_time=pl.col("time").filter(pl.col("event") == "ball_snap").min().over("game_id", "play_id"))
             .with_columns(x_std=pl.when(flip).then(120 - pl.col("x")).otherwise(pl.col("x")),
                           y_std=pl.when(flip).then(FIELD_W - pl.col("y")).otherwise(pl.col("y")),
                           dir_std=pl.when(flip).then((pl.col("dir") + 180) % 360).otherwise(pl.col("dir")),
                           o_std=pl.when(flip).then((pl.col("o") + 180) % 360).otherwise(pl.col("o")),
                           t_snap=(pl.col("time") - pl.col("snap_time")).dt.total_milliseconds() / 1000)
             .with_columns(frame_snap=(pl.col("t_snap") * 10).round(0).cast(pl.Int32),
                           x_los=pl.col("x_std") - pl.col("los_x"))      # + = downfield of LOS (offense's view)
             .sort("game_id", "play_id", "nfl_id", "time"))
aligned.sink_parquet(CACHE / "game_tracking.parquet")


Normalized combine reps

In [5]:
DRILL_ALIASES = {"LINE": "LINE_DRILL", "PASS_PRO-MIRROR_DRILL": "PASS_PRO_MIRROR_DRILL",   # renamed between years
                 "OVER_THE_SHOULDER_ADJUST": "OVER_SHOULDER_ADJUST", "SPEED_OUT_LEFT": "SPEED_OUT_ROUTE_LEFT"}
ct = pl.scan_parquet(CACHE / "combine_tracking.parquet").with_columns(pl.col("drill_name").replace(DRILL_ALIASES))
d0 = ((pl.col("x") - pl.col("x").first()) ** 2 + (pl.col("y") - pl.col("y").first()) ** 2).sqrt()
xf = (ct.filter(pl.col("entity_type") == "PLAYER").sort("event_id", "time")
        .group_by("event_id")
        .agg(x0=pl.col("x").first(), y0=pl.col("y").first(), t0=pl.col("time").first(),
             theta=pl.arctan2(pl.col("y") - pl.col("y").first(), pl.col("x") - pl.col("x").first()).filter(d0 >= 1.5).first()))
dx, dy = pl.col("x") - pl.col("x0"), pl.col("y") - pl.col("y0")
reps = (ct.join(xf, on="event_id", how="left")
          .with_columns(t_rep=(pl.col("time") - pl.col("t0")).dt.total_milliseconds() / 1000,
                        x_rep=dx * pl.col("theta").cos() + dy * pl.col("theta").sin(),
                        y_rep=-dx * pl.col("theta").sin() + dy * pl.col("theta").cos(),
                        dir_rep=(pl.col("dir") + pl.col("theta").degrees()) % 360)
          .sort("event_id", "entity_type", "time")
          .with_columns(step=((pl.col("x").diff() ** 2 + pl.col("y").diff() ** 2).sqrt()).over("event_id", "entity_type"))
          .drop("x0", "y0", "t0"))
reps.sink_parquet(CACHE / "combine_reps.parquet")

(3 reps never move 1.5 yd and get a null theta — drop them.)

Player table

In [6]:
players = (pl.read_parquet(CACHE / "players.parquet")
             .join(pl.read_parquet(CACHE / "combine_results.parquet"), on=["nfl_id", "draft_year"], how="left")
             .join(pl.read_parquet(CACHE / "player_career_successes.parquet"), on="nfl_id", how="left"))

## Sanity Checks + First Link

In [7]:
reps = pl.scan_parquet(CACHE / "combine_reps.parquet")
forty = (reps.filter(pl.col("drill_name") == "FORTY_YARD_DASH", pl.col("entity_type") == "PLAYER")
             .group_by("nfl_id").agg(combine_top_speed=pl.col("s").max()).collect())
forty.join(players, on="nfl_id").select(pl.corr("combine_top_speed", "forty"))     # expect ≈ -0.98


combine_top_speed
f64
-0.978348


In [8]:
gt = pl.scan_parquet(CACHE / "game_tracking.parquet")
game_speed = (gt.filter(pl.col("t_snap").is_between(0, 8))
                .group_by("nfl_id", "game_id", "play_id").agg(pl.col("s").max())
                .group_by("nfl_id").agg(game_top_speed=pl.col("s").quantile(0.99), n_plays=pl.len())
                .collect())
link = forty.join(game_speed, on="nfl_id").join(players.select("nfl_id", "combine_position"), on="nfl_id")
(link.filter(pl.col("n_plays") >= 100)
     .group_by("combine_position")
     .agg(pl.len(), r=pl.corr("combine_top_speed", "game_top_speed"))
     .sort("combine_position"))


combine_position,len,r
str,u32,f64
"""DB""",86,0.494185
"""DL""",77,0.697303
"""OL""",84,0.258569
"""TE""",20,0.423361
"""WR""",69,0.6045
